# E014A — vrai QArt, payload strict et blueprint adaptatif

Objectif : isoler **la construction de la condition QR** avant de modifier la diffusion. Pour chacun
des quatre prompts, la même image Stage 1 et le même bruit Stage 2 servent à comparer :

1. `binary_mask4_m` : QR v3/M/masque 4, payload exact ;
2. `qart_fragment_l` : QArt public réel, correction L, fragment `#…`, URL canonique identique ;
3. `exact_payload_mask_search_m` : meilleur des huit masques QR légaux, payload strict ;
4. `adaptive_exact_payload_m` : centres adaptés à la luminance, motifs fonctionnels binaires.

**Précision scientifique importante.** Le QArt public de `andrewyur/qart` ne peut pas être appelé
« exact-payload » : il obtient ses degrés de liberté en ajoutant un fragment. La troisième variante
est donc une recherche de masque standard strict, pas un faux QArt. Chaque sortie est testée par
OpenCV, ZBar et ZXing-cpp si disponibles, sous treize dégradations. Aucune image n'est déclarée
livrable sans porte stricte.


## Déroulement

```text
QR binaire fixe ──► DiffQRCoder Stage 1 ──► référence artistique commune
                                               │
             ┌──────────────┬──────────────────┼──────────────────┐
             ▼              ▼                  ▼                  ▼
         binaire M      QArt réel L     8 masques exacts M   adaptatif exact M
             │              │                  │                  │
             └──────── validation des blueprints + géométrie ─────┘
                                               │
                  mêmes latent initial / seed / prompt / paramètres Stage 2
                                               │
                    SSR robuste → CLIP-aes → CLIPScore → grille visible
```

Le premier lancement complet prend du temps : 4 prompts × 4 Stage 2 × 40 pas. Mettre
`PROMPT_LIMIT = 1` pour un test de plomberie, puis revenir à `None` pour l'expérience officielle.


In [ ]:
from __future__ import annotations

import gc
import hashlib
import json
import shutil
import subprocess
import sys
import time
from dataclasses import asdict
from datetime import datetime, timezone
from pathlib import Path

import lpips
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from diffusers import ControlNetModel, DDIMScheduler
from IPython.display import Markdown, clear_output, display
from PIL import Image
from safetensors.torch import load_file, save_file

UPSTREAM_ROOT = Path('/opt/DiffQRCoder')
DIFFQRCODER_COMMIT = 'e24ea73ee2e13c7e6e87cb422e8b11784e70ae00'
QART_COMMIT = '6e0e00804a1994db7098432c19fadfc552071e30'
if not (UPSTREAM_ROOT / 'diffqrcoder' / 'pipeline_diffqrcoder.py').exists():
    raise RuntimeError('DiffQRCoder absent : reconstruire Dockerfile.notebook.')
if not Path('/usr/local/bin/qart').exists():
    raise RuntimeError('QArt absent : reconstruire Dockerfile.notebook avec le stage qart-builder.')
sys.path.insert(0, str(UPSTREAM_ROOT))

from diffqrcoder import DiffQRCoderPipeline  # noqa: E402
import diffqrcoder.srpg as upstream_srpg  # noqa: E402
from prooftag_qr.blueprints import (  # noqa: E402
    align_qart_output, build_adaptive_blueprint, canonical_url_match,
    exact_mask_candidates, grid_visibility_score, reference_cost,
)
from prooftag_qr.geometry import AlignedQR, aligned_module_diagnostics, generate_aligned_qr  # noqa: E402
from prooftag_qr.quality_scoring import CLIPQualityScorer  # noqa: E402
from prooftag_qr.validation import QRValidator, summarize_validation_records  # noqa: E402


class PaperLPIPSLoss(torch.nn.Module):
    def __init__(self, *args, **kwargs):
        super().__init__()
        self.model = lpips.LPIPS(net='vgg', verbose=False)
        self.model.requires_grad_(False).eval()

    def forward(self, x, y):
        return self.model(x * 2 - 1, y * 2 - 1).mean()


upstream_srpg.PerceptualLoss = PaperLPIPSLoss
assert torch.cuda.is_available(), 'Lancer dans le pod GPU, pas avec Python Windows.'
print('GPU :', torch.cuda.get_device_name(0))
print('DiffQRCoder :', DIFFQRCODER_COMMIT)
print('QArt :', QART_COMMIT)


## 1. Contrat expérimental et dossier reprenable

In [ ]:
EXPERIMENT_NAME = 'e014a-real-qart-exact-adaptive-v1'
RESUME_RUN_NAME = None
PAYLOAD = 'https://ptag.io/t/e014'
PROMPT_LIMIT = None  # 1 = smoke test ; None = campagne officielle sur les quatre prompts
RUN_STAGE2 = True
PROMPTS = [
    {'id': 'p1_simple', 'seed': 1101, 'text': 'A single white lotus flower floating on a dark calm pond, elegant editorial photograph.'},
    {'id': 'p2_medium', 'seed': 2202, 'text': 'A Japanese garden with a red bridge, mossy stones and soft morning mist, detailed photography.'},
    {'id': 'p3_detailed', 'seed': 3303, 'text': 'An ornate botanical tapestry of white lilies, pale blue leaves and dark vines, intricate textile illustration.'},
    {'id': 'p4_complex', 'seed': 4404, 'text': 'A lively old European market square, café terraces, flowers, bicycles and a gothic cathedral, cinematic morning light.'},
]
ACTIVE_PROMPTS = PROMPTS[:PROMPT_LIMIT] if PROMPT_LIMIT else PROMPTS

QR_VERSION = 3
QR_MODULE_SIZE = 20
CANVAS_SIZE = 768
BASELINE_ECC = 'M'
BASELINE_MASK = 4
NEGATIVE_PROMPT = 'easynegative, unreadable text, letters, watermark'
BASE_MODEL_URL = 'https://huggingface.co/fp16-guy/Cetus-Mix_Whalefall_fp16_cleaned/blob/main/cetusMix_Whalefall2_fp16.safetensors'
CONTROLNET_MODEL = 'monster-labs/control_v1p_sd15_qrcode_monster'
CONTROLNET_SUBFOLDER = 'v2'
STAGE1_STEPS = 40
STAGE2_STEPS = 40
GUIDANCE_SCALE = 7.5
CONTROLNET_SCALE = 1.35
SCANNING_GUIDANCE = 500.0
PERCEPTUAL_GUIDANCE = 3.0
QART_THRESHOLDS = [96, 112, 128, 144, 160]
QART_REPEATS = 3  # le CLI public n'expose pas de seed : quantifier sa variabilité
DISPLAY_EVERY = 5
SAVE_EVERY_STEP = True
BLUEPRINT_NAMES = [
    'binary_mask4_m', 'qart_fragment_l',
    'exact_payload_mask_search_m', 'adaptive_exact_payload_m',
]

if RESUME_RUN_NAME:
    RUN_DIR = Path('/data/notebook-runs') / RESUME_RUN_NAME
    if not RUN_DIR.is_dir():
        raise FileNotFoundError(RUN_DIR)
else:
    RUN_DIR = Path('/data/notebook-runs') / (
        datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ') + '-' + EXPERIMENT_NAME
    )
    RUN_DIR.mkdir(parents=True)
RESULTS_PATH = RUN_DIR / 'results.jsonl'
print('Sortie :', RUN_DIR)
print('Exécutions Stage 2 prévues :', len(ACTIVE_PROMPTS) * 4 if RUN_STAGE2 else 0)


## 2. Fonctions d'audit : validation, reprise, frames et géométrie

In [ ]:
validator = QRValidator()
print('Décodeurs actifs :', [decoder.name for decoder in validator.decoders])
if len(validator.decoders) < 3:
    print('AVERTISSEMENT : moins de trois décodeurs. Vérifier zbar et zxing-cpp.')


def append_jsonl(path, row):
    with path.open('a', encoding='utf-8') as stream:
        stream.write(json.dumps(row, ensure_ascii=False) + '\n')
        stream.flush()


def existing_keys():
    if not RESULTS_PATH.exists():
        return set()
    return {
        (row['prompt_id'], row['blueprint'])
        for line in RESULTS_PATH.read_text(encoding='utf-8').splitlines() if line.strip()
        for row in [json.loads(line)]
    }


def validation_summary(image, mode='exact'):
    if mode == 'canonical_url':
        records = validator.validate(
            image, PAYLOAD, matcher=canonical_url_match, match_mode='canonical_url_without_fragment'
        )
    else:
        records = validator.validate(image, PAYLOAD)
    summary = summarize_validation_records(records)
    originals = [record for record in records if record.scenario == 'original']
    passed = sum(record.exact_payload_match for record in records)
    original_passed = sum(record.exact_payload_match for record in originals)
    return {
        'passed': passed, 'total': len(records), 'pass_rate': passed / len(records),
        'strict_all': passed == len(records),
        'original_passed': original_passed, 'original_total': len(originals),
        'worst_decoder_pass_rate': summary['worst_decoder_pass_rate'],
        'worst_scenario_pass_rate': summary['worst_scenario_pass_rate'],
    }, [asdict(record) for record in records]


def decode_latents(pipeline, latents):
    with torch.no_grad():
        dtype = next(pipeline.vae.parameters()).dtype
        decoded = pipeline.vae.decode(
            latents.detach().to(dtype=dtype) / pipeline.vae.config.scaling_factor,
            return_dict=False,
        )[0]
    return pipeline.image_processor.postprocess(decoded.detach(), output_type='pil')[0].convert('RGB')


def make_gif(folder, output):
    paths = sorted(folder.glob('*.jpg'))
    if not paths:
        return
    frames = [Image.open(path).convert('RGB').resize((512, 512)) for path in paths]
    frames[0].save(output, save_all=True, append_images=frames[1:], duration=150, loop=0)
    for frame in frames:
        frame.close()


def diffusion_callback(pipeline, aligned, label, steps, folder):
    folder.mkdir(parents=True, exist_ok=True)
    trace = []
    started = time.perf_counter()

    def callback(pipe_ref, step_index, timestep, callback_kwargs):
        if SAVE_EVERY_STEP or step_index % DISPLAY_EVERY == 0 or step_index == steps - 1:
            preview = decode_latents(pipeline, callback_kwargs['latents'])
            diagnostics = aligned_module_diagnostics(preview, aligned)
            row = {
                'step': int(step_index), 'timestep': int(timestep),
                'elapsed_s': time.perf_counter() - started, **diagnostics,
            }
            trace.append(row)
            preview.save(folder / f'{{step_index:03d}}.jpg', quality=88)
            if step_index % DISPLAY_EVERY == 0 or step_index == steps - 1:
                clear_output(wait=True)
                display(Markdown(
                    f"**{{label}} — {{step_index + 1}}/{{steps}} — "
                    f"MER {{diagnostics['module_error_rate']:.2%}} — "
                    f"marge {{diagnostics['minimum_threshold_margin']:.3f}}**"
                ))
                display(preview.resize((430, 430)))
        return callback_kwargs

    return callback, trace


@torch.no_grad()
def paired_stage2_latents(pipeline, stage1_tensor, seed, steps):
    normalized = stage1_tensor.to('cuda', dtype=torch.float16) * 2 - 1
    encoded = pipeline.vae.encode(normalized).latent_dist.mode() * pipeline.vae.config.scaling_factor
    generator = torch.Generator(device='cuda').manual_seed(seed)
    noise = torch.randn(encoded.shape, generator=generator, device='cuda', dtype=encoded.dtype)
    pipeline.scheduler.set_timesteps(steps, device='cuda')
    return pipeline.scheduler.add_noise(encoded, noise, pipeline.scheduler.timesteps[:1])


## 3. Charger une seule fois DiffQRCoder et figer tous ses poids

In [ ]:
controlnet = ControlNetModel.from_pretrained(
    CONTROLNET_MODEL, subfolder=CONTROLNET_SUBFOLDER, torch_dtype=torch.float16,
    cache_dir='/cache/huggingface',
)
pipe = DiffQRCoderPipeline.from_single_file(
    BASE_MODEL_URL, controlnet=controlnet, torch_dtype=torch.float16,
    cache_dir='/cache/huggingface', safety_checker=None, use_safetensors=True,
)
pipe.scheduler = DDIMScheduler.from_config(pipe.scheduler.config)
pipe = pipe.to('cuda')
for component in [pipe.unet, pipe.controlnet, pipe.vae, pipe.text_encoder]:
    component.requires_grad_(False).eval()
pipe.enable_attention_slicing('max')
pipe.enable_vae_slicing()
pipe.unet.enable_gradient_checkpointing()
pipe.controlnet.enable_gradient_checkpointing()
print('Pipeline prête ; VRAM allouée GiB :', torch.cuda.memory_allocated() / 2**30)


## 4. Stage 1 puis construction des quatre blueprints

La condition Stage 1 reste le même QR binaire v3/M/masque 4. QArt reçoit ensuite **l'image Stage 1**
comme image cible. Sa sortie 980×980 contient une bordure de dix modules : elle est recadrée sur le
cœur exact 29×29, jamais redimensionnée, puis centrée dans 768×768. Les cinq seuils QArt sont tous
validés ; le meilleur est retenu par scannabilité canonique puis proximité à la référence.


In [ ]:
def run_qart(reference_path, output_path, threshold):
    command = [
        '/usr/local/bin/qart', 'build', str(QR_VERSION), PAYLOAD,
        str(reference_path), str(output_path), '--module-size', str(QR_MODULE_SIZE),
        '--threshold', str(threshold), '--benchmark',
    ]
    completed = subprocess.run(command, check=True, text=True, capture_output=True)
    return {'command': command, 'stdout': completed.stdout, 'stderr': completed.stderr}


def save_target(prompt_dir, name, aligned, mode, extra):
    target_dir = prompt_dir / 'blueprints' / name
    target_dir.mkdir(parents=True, exist_ok=True)
    aligned.image.save(target_dir / 'condition.png')
    np.save(target_dir / 'matrix.npy', aligned.core_matrix)
    validation, records = validation_summary(aligned.image, mode)
    metadata = {
        'name': name, 'match_mode': mode, 'version': aligned.version,
        'ecc': aligned.error_correction, 'mask_pattern': aligned.mask_pattern,
        'module_size': aligned.module_size, 'padding_px': aligned.padding_px,
        'canvas_size': aligned.canvas_size, 'payload': PAYLOAD,
        'grid_visibility': grid_visibility_score(aligned.image, aligned),
        **validation, **extra,
    }
    (target_dir / 'validations.json').write_text(json.dumps(records, indent=2), encoding='utf-8')
    (target_dir / 'metadata.json').write_text(json.dumps(metadata, indent=2), encoding='utf-8')
    return {'aligned': aligned, 'metadata': metadata, 'dir': target_dir}


def load_saved_target(prompt_dir, name):
    target_dir = prompt_dir / 'blueprints' / name
    metadata = json.loads((target_dir / 'metadata.json').read_text(encoding='utf-8'))
    matrix = np.load(target_dir / 'matrix.npy').astype(np.uint8)
    image = Image.open(target_dir / 'condition.png').convert('RGB')
    aligned = AlignedQR(
        image=image, core_matrix=matrix, version=metadata['version'],
        error_correction=metadata['ecc'], mask_pattern=metadata['mask_pattern'],
        module_size=metadata['module_size'], padding_px=metadata['padding_px'],
        canvas_size=metadata['canvas_size'], payload=metadata['payload'],
    )
    return {'aligned': aligned, 'metadata': metadata, 'dir': target_dir}


stage1_states = {}
target_states = {}
for prompt_case in ACTIVE_PROMPTS:
    prompt_dir = RUN_DIR / prompt_case['id']
    prompt_dir.mkdir(parents=True, exist_ok=True)
    complete_stage1 = all(
        (prompt_dir / name).exists()
        for name in ['stage1.safetensors', 'stage1-reference.png', 'stage1-trace.json']
    )
    complete_targets = all(
        all((prompt_dir / 'blueprints' / name / artifact).exists() for artifact in [
            'condition.png', 'matrix.npy', 'metadata.json', 'validations.json',
        ])
        for name in BLUEPRINT_NAMES
    )
    if complete_stage1 and complete_targets:
        print('REPRISE artefacts figés :', prompt_case['id'])
        stage1_tensor = load_file(
            str(prompt_dir / 'stage1.safetensors'), device='cpu'
        )['stage1'].to('cuda', dtype=torch.float16)
        stage1_image = Image.open(prompt_dir / 'stage1-reference.png').convert('RGB')
        timing_path = prompt_dir / 'stage1-time.json'
        stage1_duration = (
            json.loads(timing_path.read_text(encoding='utf-8'))['duration_s']
            if timing_path.exists() else 0.0
        )
        stage1_states[prompt_case['id']] = {
            'tensor': stage1_tensor, 'image': stage1_image,
            'duration_s': stage1_duration,
        }
        target_states[prompt_case['id']] = {
            name: load_saved_target(prompt_dir, name) for name in BLUEPRINT_NAMES
        }
        continue
    if any(key[0] == prompt_case['id'] for key in existing_keys()):
        raise RuntimeError(
            f"Résultats Stage 2 présents mais artefacts blueprint incomplets pour "
            f"{prompt_case['id']}; restaurer le dossier au lieu de régénérer QArt."
        )
    baseline = generate_aligned_qr(
        PAYLOAD, version=QR_VERSION, error_correction=BASELINE_ECC,
        mask_pattern=BASELINE_MASK, module_size=QR_MODULE_SIZE, canvas_size=CANVAS_SIZE,
    )
    preflight = []
    for mask_pattern in range(8):
        candidate = generate_aligned_qr(
            PAYLOAD, version=QR_VERSION, error_correction=BASELINE_ECC,
            mask_pattern=mask_pattern, module_size=QR_MODULE_SIZE, canvas_size=CANVAS_SIZE,
        )
        validation, _ = validation_summary(candidate.image, 'exact')
        preflight.append((validation, candidate))
    preflight.sort(
        key=lambda item: (
            item[0]['strict_all'], item[0]['pass_rate'],
            item[0]['worst_decoder_pass_rate'], item[0]['worst_scenario_pass_rate'],
            item[1].mask_pattern == BASELINE_MASK, -item[1].mask_pattern,
        ),
        reverse=True,
    )
    if not preflight[0][0]['strict_all']:
        raise RuntimeError(
            f"Aucun QR témoin v{QR_VERSION}/{BASELINE_ECC} ne passe la porte stricte ; "
            "changer explicitement payload/version/ECC avant toute diffusion."
        )
    stage1_control = preflight[0][1]
    (prompt_dir / 'stage1-control-preflight.json').write_text(
        json.dumps([
            {'mask': item[1].mask_pattern, **item[0]} for item in preflight
        ], indent=2), encoding='utf-8'
    )
    stage1_control.image.save(prompt_dir / 'stage1-control.png')
    stage1_folder = prompt_dir / 'frames-stage1'
    callback, trace = diffusion_callback(
        pipe, stage1_control, f"{{prompt_case['id']}} / Stage 1", STAGE1_STEPS, stage1_folder
    )
    started = time.perf_counter()
    result = pipe._run_stage1(
        prompt=prompt_case['text'], qrcode=stage1_control.image, negative_prompt=NEGATIVE_PROMPT,
        num_inference_steps=STAGE1_STEPS, guidance_scale=GUIDANCE_SCALE,
        generator=torch.Generator(device='cuda').manual_seed(prompt_case['seed']),
        controlnet_conditioning_scale=CONTROLNET_SCALE,
        callback_on_step_end=callback, callback_on_step_end_tensor_inputs=['latents'],
        output_type='pt',
    )
    stage1_tensor = result.images.detach()
    stage1_image = pipe.image_processor.numpy_to_pil(
        pipe.image_processor.pt_to_numpy(stage1_tensor.detach())
    )[0].convert('RGB')
    stage1_image.save(prompt_dir / 'stage1-reference.png')
    save_file({'stage1': stage1_tensor.cpu().contiguous()}, str(prompt_dir / 'stage1.safetensors'))
    (prompt_dir / 'stage1-trace.json').write_text(json.dumps(trace, indent=2), encoding='utf-8')
    make_gif(stage1_folder, prompt_dir / 'stage1.gif')
    stage1_duration = time.perf_counter() - started
    (prompt_dir / 'stage1-time.json').write_text(
        json.dumps({'duration_s': stage1_duration}, indent=2), encoding='utf-8'
    )
    stage1_states[prompt_case['id']] = {
        'tensor': stage1_tensor, 'image': stage1_image,
        'duration_s': stage1_duration,
    }

    targets = {}
    targets['binary_mask4_m'] = save_target(
        prompt_dir, 'binary_mask4_m', baseline, 'exact',
        {'algorithm': 'standard QR, fixed mask 4', 'reference_cost': reference_cost(baseline.image, stage1_image)},
    )

    exact_candidates = exact_mask_candidates(
        PAYLOAD, stage1_image, version=QR_VERSION, error_correction=BASELINE_ECC,
        module_size=QR_MODULE_SIZE, canvas_size=CANVAS_SIZE,
    )
    exact_rows = []
    for candidate in exact_candidates:
        validation, _ = validation_summary(candidate.aligned.image, 'exact')
        exact_rows.append((validation, candidate))
    exact_rows.sort(
        key=lambda item: (
            item[0]['strict_all'], item[0]['pass_rate'],
            item[0]['worst_decoder_pass_rate'], item[0]['worst_scenario_pass_rate'],
            -item[1].reference_cost, -item[1].grid_visibility,
        ),
        reverse=True,
    )
    exact_winner = exact_rows[0][1].aligned
    targets['exact_payload_mask_search_m'] = save_target(
        prompt_dir, 'exact_payload_mask_search_m', exact_winner, 'exact',
        {'algorithm': 'best of eight legal QR masks; not QArt',
         'reference_cost': exact_rows[0][1].reference_cost,
         'all_mask_costs': [
             {
                 'mask': item[1].aligned.mask_pattern,
                 'reference_cost': item[1].reference_cost,
                 **item[0],
             }
             for item in exact_rows
         ]},
    )

    adaptive_rows = []
    for minimum_fraction in [0.22, 0.30, 0.38, 0.46, 0.55, 0.70, 0.85]:
        adaptive = build_adaptive_blueprint(
            stage1_image, exact_winner, minimum_data_fraction=minimum_fraction
        )
        validation, _ = validation_summary(adaptive.image, 'exact')
        adaptive_rows.append((validation, adaptive, minimum_fraction))
    adaptive_rows.sort(
        key=lambda item: (
            item[0]['strict_all'], item[0]['pass_rate'],
            -item[1].reference_cost, -item[1].grid_visibility,
        ),
        reverse=True,
    )
    adaptive_validation, adaptive, adaptive_fraction = adaptive_rows[0]
    adaptive_aligned = AlignedQR(
        image=adaptive.image, core_matrix=exact_winner.core_matrix.copy(),
        version=QR_VERSION, error_correction=BASELINE_ECC,
        mask_pattern=exact_winner.mask_pattern, module_size=QR_MODULE_SIZE,
        padding_px=exact_winner.padding_px, canvas_size=CANVAS_SIZE, payload=PAYLOAD,
    )
    np.save(prompt_dir / 'adaptive-center-fractions.npy', adaptive.center_fractions)
    targets['adaptive_exact_payload_m'] = save_target(
        prompt_dir, 'adaptive_exact_payload_m', adaptive_aligned, 'exact',
        {'algorithm': 'Prooftag luminance-adaptive centers; functional patterns binary',
         'minimum_data_fraction': adaptive_fraction,
         'reference_cost': adaptive.reference_cost,
         'screened_minimum_fractions': [row[2] for row in adaptive_rows]},
    )

    qart_rows = []
    qart_errors = []
    for threshold in QART_THRESHOLDS:
        for repeat in range(QART_REPEATS):
            raw_path = prompt_dir / f'qart-raw-threshold-{{threshold}}-repeat-{{repeat}}.png'
            try:
                provenance = run_qart(prompt_dir / 'stage1-reference.png', raw_path, threshold)
                aligned_qart = align_qart_output(
                    Image.open(raw_path), payload=PAYLOAD, version=QR_VERSION,
                    module_size=QR_MODULE_SIZE, canvas_size=CANVAS_SIZE,
                )
                validation, _ = validation_summary(aligned_qart.image, 'canonical_url')
                qart_rows.append({
                    'validation': validation, 'aligned': aligned_qart,
                    'threshold': threshold, 'repeat': repeat,
                    'reference_cost': reference_cost(aligned_qart.image, stage1_image),
                    'provenance': provenance,
                    'sha256': hashlib.sha256(raw_path.read_bytes()).hexdigest(),
                })
            except Exception as exc:
                qart_errors.append({
                    'threshold': threshold, 'repeat': repeat,
                    'error': f'{{type(exc).__name__}}: {{exc}}',
                })
    (prompt_dir / 'qart-errors.json').write_text(
        json.dumps(qart_errors, indent=2), encoding='utf-8'
    )
    if not qart_rows:
        raise RuntimeError('Tous les essais QArt ont échoué ; voir qart-errors.json.')
    qart_rows.sort(
        key=lambda row: (
            row['validation']['strict_all'], row['validation']['pass_rate'],
            -row['reference_cost'],
        ),
        reverse=True,
    )
    qart_winner = qart_rows[0]
    targets['qart_fragment_l'] = save_target(
        prompt_dir, 'qart_fragment_l', qart_winner['aligned'], 'canonical_url',
        {'algorithm': 'andrewyur/qart real Reed-Solomon degrees of freedom',
         'exact_payload': False, 'canonical_url_only': True,
         'threshold': qart_winner['threshold'], 'repeat': qart_winner['repeat'],
         'reference_cost': qart_winner['reference_cost'],
         'raw_sha256': qart_winner['sha256'], 'provenance': qart_winner['provenance']},
    )
    target_states[prompt_case['id']] = targets

    blueprint_rows = [state['metadata'] for state in targets.values()]
    pd.DataFrame(blueprint_rows).to_csv(prompt_dir / 'blueprint-comparison.csv', index=False)
    display(pd.DataFrame(blueprint_rows)[
        ['name', 'ecc', 'match_mode', 'passed', 'total', 'reference_cost', 'grid_visibility']
    ])


## 5. Comparaison Stage 2 appariée

Les quatre branches réutilisent le même tensor Stage 1, le même latent initial DDIM, la même seed,
le même prompt et les mêmes poids. Seule l'image de condition QR change. Une frame décodée et ses
diagnostics sont écrits à chaque pas. Le QArt est évalué en URL canonique ; les trois autres en
égalité stricte du payload.


In [ ]:
quality_scorer = CLIPQualityScorer(Path('/cache'), device='cpu')


def rank_row(row):
    return (
        row['strict_all'], row['pass_rate'], row['worst_decoder_pass_rate'],
        row['worst_scenario_pass_rate'], row.get('clip_aesthetic') or -999,
        row.get('clip_score') or -999, -row['grid_visibility'],
    )


if RUN_STAGE2:
    for prompt_case in ACTIVE_PROMPTS:
        prompt_id = prompt_case['id']
        stage1 = stage1_states[prompt_id]
        for blueprint_name, state in target_states[prompt_id].items():
            if (prompt_id, blueprint_name) in existing_keys():
                print('SKIP', prompt_id, blueprint_name)
                continue
            aligned = state['aligned']
            output_dir = RUN_DIR / prompt_id / 'stage2' / blueprint_name
            output_dir.mkdir(parents=True, exist_ok=True)
            callback, trace = diffusion_callback(
                pipe, aligned, f"{{prompt_id}} / {{blueprint_name}}",
                STAGE2_STEPS, output_dir / 'frames',
            )
            initial = paired_stage2_latents(
                pipe, stage1['tensor'], prompt_case['seed'] + 10000, STAGE2_STEPS
            )
            started = time.perf_counter()
            result = pipe._run_stage2(
                prompt=prompt_case['text'], qrcode=aligned.image,
                qrcode_module_size=aligned.module_size, qrcode_padding=aligned.padding_px,
                ref_image=stage1['tensor'], negative_prompt=NEGATIVE_PROMPT,
                num_inference_steps=STAGE2_STEPS, guidance_scale=GUIDANCE_SCALE, eta=0.0,
                generator=torch.Generator(device='cuda').manual_seed(prompt_case['seed'] + 10000),
                latents=initial.clone(), controlnet_conditioning_scale=CONTROLNET_SCALE,
                scanning_robust_guidance_scale=SCANNING_GUIDANCE,
                perceptual_guidance_scale=PERCEPTUAL_GUIDANCE,
                callback_on_step_end=callback, callback_on_step_end_tensor_inputs=['latents'],
                output_type='latent',
            )
            final_latent = result.images.detach()
            image = decode_latents(pipe, final_latent)
            duration = time.perf_counter() - started
            image.save(output_dir / 'final.png')
            save_file(
                {'latents': final_latent.cpu().contiguous()},
                str(output_dir / 'final-latent.safetensors'),
            )
            (output_dir / 'trace.json').write_text(json.dumps(trace, indent=2), encoding='utf-8')
            make_gif(output_dir / 'frames', output_dir / 'diffusion.gif')
            mode = state['metadata']['match_mode']
            validation, records = validation_summary(image, mode)
            (output_dir / 'validations.json').write_text(
                json.dumps(records, indent=2), encoding='utf-8'
            )
            try:
                quality = asdict(quality_scorer.score(image, prompt_case['text']))
                quality_error = None
            except Exception as exc:
                quality = {'clip_similarity': None, 'clip_score': None, 'clip_aesthetic': None}
                quality_error = f'{{type(exc).__name__}}: {{exc}}'
            row = {
                'prompt_id': prompt_id, 'prompt': prompt_case['text'],
                'seed': prompt_case['seed'], 'blueprint': blueprint_name,
                'payload_contract': mode, 'stage1_duration_s': stage1['duration_s'],
                'stage2_duration_s': duration,
                **validation, **quality, **aligned_module_diagnostics(image, aligned),
                'grid_visibility': grid_visibility_score(image, aligned),
                'blueprint_reference_cost': state['metadata']['reference_cost'],
                'quality_error': quality_error,
            }
            append_jsonl(RESULTS_PATH, row)
            print(prompt_id, blueprint_name, validation['passed'], '/', validation['total'])
            del result, final_latent, initial
            gc.collect()
            torch.cuda.empty_cache()


## 6. Décision, manifeste et archive

In [ ]:
rows = [
    json.loads(line) for line in RESULTS_PATH.read_text(encoding='utf-8').splitlines()
    if line.strip()
] if RESULTS_PATH.exists() else []
frame = pd.DataFrame(rows)
if not frame.empty:
    frame.to_csv(RUN_DIR / 'comparison.csv', index=False)
    display(frame.sort_values(
        ['strict_all', 'pass_rate', 'clip_aesthetic', 'clip_score'],
        ascending=[False, False, False, False],
    )[
        ['prompt_id', 'blueprint', 'passed', 'total', 'clip_aesthetic',
         'clip_score', 'grid_visibility', 'stage2_duration_s']
    ])

    aggregate = frame.groupby('blueprint').agg(
        contexts=('prompt_id', 'nunique'),
        strict_contexts=('strict_all', 'sum'),
        mean_ssr=('pass_rate', 'mean'),
        worst_ssr=('pass_rate', 'min'),
        mean_aesthetic=('clip_aesthetic', 'mean'),
        mean_clip=('clip_score', 'mean'),
        mean_grid_visibility=('grid_visibility', 'mean'),
        mean_seconds=('stage2_duration_s', 'mean'),
    ).reset_index()
    aggregate.to_csv(RUN_DIR / 'aggregate.csv', index=False)
    display(aggregate.sort_values(
        ['strict_contexts', 'worst_ssr', 'mean_ssr', 'mean_aesthetic'],
        ascending=False,
    ))

    for prompt_case in ACTIVE_PROMPTS:
        prompt_rows = [row for row in rows if row['prompt_id'] == prompt_case['id']]
        exact_rows = [row for row in prompt_rows if row['payload_contract'] == 'exact']
        winner = max(exact_rows, key=rank_row)
        source = RUN_DIR / prompt_case['id'] / 'blueprints' / winner['blueprint']
        destination = RUN_DIR / prompt_case['id']
        shutil.copy2(source / 'condition.png', destination / 'selected-blueprint.png')
        shutil.copy2(source / 'matrix.npy', destination / 'selected-matrix.npy')
        selected = {
            'prompt_id': prompt_case['id'], 'prompt': prompt_case['text'],
            'payload': PAYLOAD, 'selected_blueprint': winner['blueprint'],
            'selection_rule': 'exact payload only; strict SSR then weak links then aesthetics',
            'module_size': QR_MODULE_SIZE, 'padding_px': target_states[prompt_case['id']][winner['blueprint']]['aligned'].padding_px,
            'canvas_size': CANVAS_SIZE, 'version': QR_VERSION,
            'stage1_tensor': str(destination / 'stage1.safetensors'),
            'blueprint_path': str(destination / 'selected-blueprint.png'),
            'matrix_path': str(destination / 'selected-matrix.npy'),
        }
        (destination / 'selected-meta.json').write_text(
            json.dumps(selected, indent=2), encoding='utf-8'
        )

manifest = {
    'experiment': EXPERIMENT_NAME, 'created_at': datetime.now(timezone.utc).isoformat(),
    'diffqrcoder_commit': DIFFQRCODER_COMMIT, 'qart_commit': QART_COMMIT,
    'payload': PAYLOAD, 'prompts': ACTIVE_PROMPTS, 'canvas_size': CANVAS_SIZE,
    'module_size': QR_MODULE_SIZE, 'stage1_steps': STAGE1_STEPS,
    'stage2_steps': STAGE2_STEPS, 'decoders': [decoder.name for decoder in validator.decoders],
    'scientific_limits': [
        'QArt public appends a fragment and uses ECC L; it is canonical-URL, not exact-payload.',
        'QArt public exposes no deterministic seed; repeated outputs and SHA-256 are persisted.',
        'Exact-payload mask search uses standard QR masks and is not labelled QArt.',
        'Adaptive blueprint is a Prooftag engineering method, not a paper reproduction.',
        'Software validation is not physical SSR.',
    ],
}
(RUN_DIR / 'manifest.json').write_text(json.dumps(manifest, indent=2), encoding='utf-8')
archive = shutil.make_archive(
    str(RUN_DIR), 'gztar', root_dir=RUN_DIR.parent, base_dir=RUN_DIR.name
)
print('Archive :', archive)
